In [ ]:
import sys

print(sys.executable)

/home/nicolas/laboratorio_pinguinos/.venv/bin/python


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
# Cargar el dataset
df = sns.load_dataset('penguins')


### Fase 1: Enfoque clásico

#### PASO 1: Análisis exploratorio de datos

**Observación inicial:**

1. ¿Cuántas filas y columnas tiene el dataset?  
2. ¿Qué variables son numéricas y cuáles categóricas?  
3. ¿Cuántos valores faltantes hay por columna?  
4. ¿Existen filas duplicadas?  
5. ¿Qué variables tienen baja cardinalidad?  

En el siguiente codigo vamos a responder las preguntas anteriores:

* vamos a utilizar 'shape' que me devuelve una tupla del número de columnas y filas.
* la funcion select_types la cual me selecciona las columnas que contienen valores numericos y no numericos (str) ademas el tolist me convierte el objeto que escupe select_types a una lista de python la cual me permite navergarla mejor.
* is_null() me dice cuando las variables son nulas y sum() me las agrupa y las suma.
* duplicated() me avisa si hay una fila doble 
 

In [ ]:
# 1. ¿Cuántas filas y columnas tiene el dataset?
num_rows, num_columns = df.shape
print(f"El dataset tiene {num_rows} filas y {num_columns} columnas.")

# 2. ¿Qué variables son numéricas y cuáles categóricas?
numeric_vars = df.select_dtypes(include=['number']).columns.tolist()
categorical_vars = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Variables numéricas: {numeric_vars}")
print(f"Variables categóricas: {categorical_vars}")

# 3. ¿Cuántos valores faltantes hay por columna?
missing_values = df.isnull().sum()
print("Valores faltantes por columna:")
print(missing_values)

# 4. ¿Existen filas duplicadas?
duplicates = df.duplicated().sum()
print(f"Existen {duplicates} filas duplicadas.")

# 5. ¿Qué variables tienen baja cardinalidad?
low_cardinality_vars = [col for col in categorical_vars if df[col].nunique() < 4]
print(f"Variables con baja cardinalidad: {low_cardinality_vars}")

El dataset tiene 344 filas y 7 columnas.
Variables numéricas: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
Variables categóricas: ['species', 'island', 'sex']
Valores faltantes por columna:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64
Existen 0 filas duplicadas.
Variables con baja cardinalidad: ['species', 'island', 'sex']


/tmp/ipykernel_102995/444680291.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_vars = df.select_dtypes(include=['object', 'category']).columns.tolist()


### Parte B — Descripción

6. Para cada variable numérica, reporte:  
   - Media  
   - Mediana  
   - Desviación estándar  
   - Rango intercuartílico  

7. Para cada variable categórica, reporte:  
   - Conteos  
   - Porcentajes  

8. Construya tablas cruzadas para pares de variables categóricas relevantes.  

9. Calcule matrices de correlación (Pearson y Spearman) entre variables numéricas.  

### Funciones utilizadas

- `mean()`: calcula la media de cada variable numérica.
- `median()`: obtiene la mediana de cada variable numérica.
- `std()`: calcula la desviación estándar.
- `quantile()`: obtiene los cuartiles; al restar el primer cuartil del tercero se calcula el rango intercuartílico.
- `value_counts()`: cuenta la frecuencia de cada categoría.
- `value_counts(normalize=True)`: calcula la proporción de cada categoría, que luego se expresa como porcentaje.
- `crosstab()`: construye tablas cruzadas entre pares de variables categóricas.
- `corr()`: calcula matrices de correlación usando los métodos de Pearson y Spearman.
- `display()`: muestra los resultados como tablas dentro del notebook.
- `dropna=False`: permite incluir los valores faltantes en los conteos y tablas cruzadas.

In [ ]:
from itertools import combinations
from IPython.display import display

# 6. Estadísticas descriptivas de las variables numéricas
estadisticas_numericas = pd.DataFrame({
    "Media": df[numeric_vars].mean(),
    "Mediana": df[numeric_vars].median(),
    "Desviación estándar": df[numeric_vars].std(),
    "Rango intercuartílico": df[numeric_vars].quantile(0.75) - df[numeric_vars].quantile(0.25)
})

print("6. Estadísticas descriptivas de variables numéricas")
display(estadisticas_numericas)

# 7. Conteos y porcentajes de cada variable categórica
print("7. Conteos y porcentajes de variables categóricas")
for variable in categorical_vars:
    conteos = df[variable].value_counts(dropna=False)
    porcentajes = df[variable].value_counts(normalize=True, dropna=False).mul(100)
    resumen_categorico = pd.DataFrame({
        "Conteo": conteos,
        "Porcentaje (%)": porcentajes.round(2)
    })
    print(f"\\nVariable: {variable}")
    display(resumen_categorico)

# 8. Tablas cruzadas para todos los pares de variables categóricas
print("8. Tablas cruzadas de variables categóricas")
for variable_a, variable_b in combinations(categorical_vars, 2):
    print(f"\\n{variable_a} x {variable_b} (conteos)")
    display(pd.crosstab(df[variable_a], df[variable_b], dropna=False))

    print(f"{variable_a} x {variable_b} (porcentajes por fila)")
    display(pd.crosstab(
        df[variable_a],
        df[variable_b],
        normalize="index",
        dropna=False
    ).mul(100).round(2))

# 9. Matrices de correlación entre variables numéricas
print("9. Matriz de correlación de Pearson")
display(df[numeric_vars].corr(method="pearson").round(3))

print("Matriz de correlación de Spearman")
display(df[numeric_vars].corr(method="spearman").round(3))

6. Estadísticas descriptivas de variables numéricas


,Media,Mediana,Desviación estándar,Rango intercuartílico
bill_length_mm,43.921930,44.45,5.459584,9.275
bill_depth_mm,17.151170,17.30,1.974793,3.100
flipper_length_mm,200.915205,197.00,14.061714,23.000
body_mass_g,4201.754386,4050.00,801.954536,1200.000


7. Conteos y porcentajes de variables categóricas
\nVariable: species


,Conteo,Porcentaje (%)
species,,
Adelie,152,44.19
Gentoo,124,36.05
Chinstrap,68,19.77


\nVariable: island


...

sex,Female,Male,NaN
island,,,
Biscoe,47.62,49.40,2.98
Dream,49.19,50.00,0.81
Torgersen,46.15,44.23,9.62


9. Matriz de correlación de Pearson


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
bill_length_mm,1.000,-0.235,0.656,0.595
bill_depth_mm,-0.235,1.000,-0.584,-0.472
flipper_length_mm,0.656,-0.584,1.000,0.871
body_mass_g,0.595,-0.472,0.871,1.000


Matriz de correlación de Spearman


,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
bill_length_mm,1.000,-0.222,0.673,0.584
bill_depth_mm,-0.222,1.000,-0.523,-0.432
flipper_length_mm,0.673,-0.523,1.000,0.840
body_mass_g,0.584,-0.432,0.840,1.000


### Parte C — Visualización

10. Genere gráficos de conteo para todas las variables categóricas de baja cardinalidad.
11. Genere histogramas para las variables numéricas y describa su forma.
12. Genere un boxplot de `bill_length_mm` por `species`.
13. Genere un scatter entre `bill_length_mm` y `bill_depth_mm`, coloreado por `species`.
14. Genere un heatmap de correlación entre las variables numéricas.

Los gráficos permiten complementar las tablas descriptivas: los conteos muestran el balance de las categorías, los histogramas permiten observar la forma y la dispersión, el boxplot compara distribuciones entre grupos, el scatter muestra relaciones entre dos variables y el heatmap resume las correlaciones.

In [ ]:
# 10. Gráficos de conteo para variables categóricas de baja cardinalidad
if low_cardinality_vars:
    filas = (len(low_cardinality_vars) + 1) // 2
    fig, axes = plt.subplots(filas, 2, figsize=(14, 5 * filas))
    axes = np.atleast_1d(axes).ravel()

    for eje, variable in zip(axes, low_cardinality_vars):
        sns.countplot(data=df, x=variable, order=df[variable].value_counts().index, ax=eje)
        eje.set_title(f"Conteo de {variable}")
        eje.set_xlabel(variable)
        eje.set_ylabel("Frecuencia")
        eje.tick_params(axis="x", rotation=20)

    for eje in axes[len(low_cardinality_vars):]:
        eje.remove()
    plt.tight_layout()
    plt.show()
else:
    print("No se encontraron variables categóricas de baja cardinalidad.")

# 11. Histogramas y descripción de la forma de las variables numéricas
fig, axes = plt.subplots(len(numeric_vars), 1, figsize=(10, 4 * len(numeric_vars)))
axes = np.atleast_1d(axes)

for eje, variable in zip(axes, numeric_vars):
    sns.histplot(data=df, x=variable, kde=True, bins=20, ax=eje)
    eje.set_title(f"Histograma de {variable}")
    eje.set_xlabel(variable)
    eje.set_ylabel("Frecuencia")

    asimetria = df[variable].dropna().skew()
    if abs(asimetria) < 0.2:
        forma = "aproximadamente simétrica"
    elif asimetria > 0:
        forma = "asimétrica positiva (cola hacia la derecha)"
    else:
        forma = "asimétrica negativa (cola hacia la izquierda)"
    print(f"{variable}: distribución {forma}; asimetría = {asimetria:.2f}.")

plt.tight_layout()
plt.show()

# 12. Boxplot de una variable numérica por categoría
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x="species", y="bill_length_mm", hue="species", legend=False)
plt.title("Longitud del pico por especie")
plt.xlabel("Especie")
plt.ylabel("Longitud del pico (mm)")
plt.tight_layout()
plt.show()

# 13. Relación entre dos variables numéricas coloreada por categoría
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df,
    x="bill_length_mm",
    y="bill_depth_mm",
    hue="species",
    s=70,
    alpha=0.8
)
plt.title("Relación entre longitud y profundidad del pico")
plt.xlabel("Longitud del pico (mm)")
plt.ylabel("Profundidad del pico (mm)")
plt.tight_layout()
plt.show()

# 14. Heatmap de correlación entre variables numéricas
plt.figure(figsize=(10, 7))
correlacion = df[numeric_vars].corr(method="pearson")
sns.heatmap(correlacion, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Matriz de correlación de Pearson")
plt.tight_layout()
plt.show()